In [1]:
# ==================================================
# 0) Imports & Hyper‑params
# ==================================================
import re, random, math, time, itertools, os
from math import ceil
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==================================================
# Adjust these!
# ==================================================
VOCAB_LIMIT = 50_000
MAX_LEN     = 512
BATCH       = 32
SEED        = 42
EPOCHS      = 5
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
# --------------------------------------------------

if DEVICE == "cuda":
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
    torch.backends.cudnn.benchmark = True

In [2]:
# ==================================================
# 1) “basic_english” tokenizer drop‑in
# ==================================================
_basic_english_re = re.compile(
    r"""([!"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~])   # any punctuation
     |(\d+[%]?)                                  # numbers (and percent)
     |([A-Za-z]+(?:'[A-Za-z]+)?)                 # words w/ optional apos
    """,
    re.VERBOSE,
)
def basic_english_tokenizer(text: str) -> list[str]:
    text = text.lower()
    tokens = []
    for punc, num, word in _basic_english_re.findall(text):
        if punc:   tokens.append(punc)
        elif num:  tokens.append(num)
        elif word: tokens.append(word)
    return tokens

tok = basic_english_tokenizer

In [3]:
# ==================================================
# 2) EDA augmenters
# ==================================================
def eda_random_deletion(tokens, p=0.05):
    if len(tokens) == 1:
        return tokens
    out = [t for t in tokens if random.random() > p]
    return out or [random.choice(tokens)]

def eda_random_swap(tokens, n_swaps=3):
    toks = tokens.copy()
    for _ in range(n_swaps):
        i, j = random.sample(range(len(toks)), 2)
        toks[i], toks[j] = toks[j], toks[i]
    return toks

In [4]:
# ==================================================
# 3) Load IMDB and build vocab
# ==================================================
raw = load_dataset("imdb")
train_examples = list(zip(raw["train"]["label"], raw["train"]["text"]))
test_examples  = list(zip(raw["test"]["label"],  raw["test"]["text"]))

counter = Counter()
for lbl, txt in train_examples:
    counter.update(tok(txt))
most_common = [w for w,_ in counter.most_common(VOCAB_LIMIT)]
stoi = {w:i+2 for i,w in enumerate(most_common)}
stoi["<pad>"] = 0; stoi["<unk>"] = 1
PAD_IDX, UNK_IDX = 0, 1
VOCAB_SIZE = len(stoi)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
# ==================================================
# 4) AugmentedIMDB Dataset
# ==================================================
class AugmentedIMDB(Dataset):
    def __init__(self, examples, max_len, augment=False):
        self.examples = examples
        self.max_len  = max_len
        self.augment  = augment

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        lbl, txt = self.examples[idx]
        toks = tok(txt)
        if self.augment:
            op = random.choice(["del","swap",None])
            if   op=="del":  toks = eda_random_deletion(toks)
            elif op=="swap": toks = eda_random_swap(toks)
        toks = toks[: self.max_len]
        ids  = [stoi.get(t, UNK_IDX) for t in toks]
        if len(ids) < self.max_len:
            ids += [PAD_IDX] * (self.max_len - len(ids))
        return lbl, torch.tensor(ids, dtype=torch.long)

    def collate_fn(self, batch):
        labels, texts = zip(*batch)
        texts = torch.stack(texts, dim=0)
        masks = (texts != PAD_IDX).long()
        return texts, masks, torch.tensor(labels, dtype=torch.long)

In [6]:
# ==============================================================
# 5) Exactly‑copied “long” splitting with deterministic seeding
# ==============================================================
def make_long_subsets(examples):
    random.seed(SEED)
    real_long, short_pos, short_neg, super_long = [], [], [], []
    for lbl, txt in examples:
        L = len(tok(txt))
        if MAX_LEN <= L < 2 * MAX_LEN:
            real_long.append((lbl, txt))
        elif L >= 2 * MAX_LEN:
            super_long.append((lbl, txt))
        else:
            (short_pos if lbl == 1 else short_neg).append(txt)

    split_long = []
    for lbl, txt in super_long:
        toks = tok(txt)
        step = MAX_LEN // 2
        num_windows = ceil((len(toks) - MAX_LEN) / step) + 1
        for w in range(num_windows):
            start = min(w * step, len(toks) - MAX_LEN)
            window = toks[start : start + MAX_LEN]
            split_long.append((lbl, " ".join(window)))
            if start + MAX_LEN >= len(toks):
                break

    random.seed(SEED)
    random.shuffle(short_pos)
    random.seed(SEED)
    random.shuffle(short_neg)

    new_long = []
    for pool, label in [(short_pos,1),(short_neg,0)]:
        i = 0
        while i + 1 < len(pool):
            combo = tok(pool[i]) + tok(pool[i+1])
            if len(combo) >= MAX_LEN:
                new_long.append((label, " ".join(combo)))
                i += 2
            else:
                if i + 2 < len(pool):
                    combo3 = combo + tok(pool[i+2])
                    if len(combo3) >= MAX_LEN:
                        new_long.append((label, " ".join(combo3)))
                        i += 3; continue
                i += 1

    long_train = real_long + split_long + new_long
    random.seed(SEED); random.shuffle(long_train)

    long_test = [(lbl,txt) for lbl,txt in examples if len(tok(txt)) >= MAX_LEN]
    random.seed(SEED); random.shuffle(long_test)

    return long_train, long_test

In [7]:
# ==================================================
# 6) DataLoaders
# ==================================================
def get_data():
    long_train, _ = make_long_subsets(train_examples)
    print(f"--> FINAL long_train: {len(long_train)}")

    _, long_test   = make_long_subsets(test_examples)
    print(f"--> FINAL long_test:  {len(long_test)}")
    train_ds = AugmentedIMDB(long_train, MAX_LEN, augment=True)
    train_dl = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True, drop_last=True,
        pin_memory=(DEVICE == "cuda"), num_workers=4,
        collate_fn=train_ds.collate_fn,
        generator=torch.Generator().manual_seed(SEED),
    )
    test_ds = AugmentedIMDB(long_test, MAX_LEN, augment=False)
    test_dl = DataLoader(
        test_ds, batch_size=BATCH, shuffle=False,
        pin_memory=(DEVICE == "cuda"), num_workers=2,
        collate_fn=test_ds.collate_fn,
    )
    return train_dl, test_dl

In [8]:
# ==================================================
# 7) Attention blocks (pad‑mask aware)
# ==================================================
class MultiHeadAttention(nn.Module):
    def __init__(self, d, h, drop, qkv_bias=False):
        super().__init__()
        self.h, self.dk = h, d//h
        self.q = nn.Linear(d,d, bias=qkv_bias)
        self.k = nn.Linear(d,d, bias=qkv_bias)
        self.v = nn.Linear(d,d, bias=qkv_bias)
        self.o = nn.Linear(d,d)
        self.drop = nn.Dropout(drop)

    def forward(self, x, mask):
        B,T,_ = x.shape
        Q = self.q(x).view(B,T,self.h,self.dk).transpose(1,2)
        K = self.k(x).view(B,T,self.h,self.dk).transpose(1,2)
        V = self.v(x).view(B,T,self.h,self.dk).transpose(1,2)

        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.dk)
        if mask is not None:
            pad = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(pad==0, float("-inf"))
        W = torch.softmax(scores, -1)
        W = self.drop(W)
        out = (W @ V).transpose(1,2).contiguous().view(B,T,self.h*self.dk)
        return self.o(out)


class AngularAttention(nn.Module):
    def __init__(self, d, h, drop, qkv_bias=False):
        super().__init__()
        self.h, self.dk = h, d//h
        self.q = nn.Linear(d,d, bias=qkv_bias)
        self.k = nn.Linear(d,d, bias=qkv_bias)
        self.v = nn.Linear(d,d, bias=qkv_bias)
        self.o = nn.Linear(d,d)
        self.drop = nn.Dropout(drop)

    def forward(self, x, mask):
        B,T,_ = x.shape
        Q = F.normalize(self.q(x).view(B,T,self.h,self.dk).transpose(1,2), dim=-1)
        K = F.normalize(self.k(x).view(B,T,self.h,self.dk).transpose(1,2), dim=-1)
        V = self.v(x).view(B,T,self.h,self.dk).transpose(1,2)
        sim = (Q @ K.transpose(-2,-1)).clamp(-0.999,0.999)
        scores = 1 - torch.acos(sim)/math.pi
        if mask is not None:
            pad = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(pad==0, 0.0)
        W = scores.clamp(min=1e-6).pow(18)
        W = W / (W.sum(-1,keepdim=True)+1e-6)
        W = self.drop(W)
        out = (W @ V).transpose(1,2).contiguous().view(B,T,self.h*self.dk)
        return self.o(out)

# Different

class BatchedACE(nn.Module):
    """
    Causal (LM) BatchedACE that optionally uses a single shared sketch
    or independent sketches per ensemble.
    Inputs:
      Khf, Vhf, Qhf : [M, B, T, H, d_k]
    """
    def __init__(self, d_k, K, L, M, device='cpu', share_planes: bool = True):
        super().__init__()
        self.d_k, self.K, self.L, self.M = d_k, K, L, M
        self.R = 1 << K
        self.share_planes = share_planes

        if share_planes:
            # Shared planes [L, K, d_k] --> [d_k, (L*K)]
            planes = torch.randn(L, K, d_k, device=device)
            self.register_buffer(
              'planes_T',
              planes.view(L*K, d_k).T
            )  # [d_k, L*K]
        else:
            # Independent planes [M, L, K, d_k] --> [M, d_k, (L*K)]
            planes = torch.randn(M, L, K, d_k, device=device)
            planes = planes.view(M, L*K, d_k).transpose(1,2)
            self.register_buffer('planes_T', planes)
            # planes_T: [M, d_k, L*K]

        # flatten protos [R, K] --> [K, R]
        corners = torch.tensor(
            list(itertools.product([-1., +1.], repeat=K)),
            device=device
        )
        self.register_buffer('protos_T', corners.T)  # [K, R]

        # # learnable temperature
        # self.logit_temp = nn.Parameter(torch.log(torch.tensor(1.0)))

    def forward(self, Khf, Vhf, Qhf):
        # [M, B, T, H, d_k]
        M, B, T, H, dk = Khf.shape
        assert M == self.M and dk == self.d_k
        scale = math.sqrt(dk)
        # scale = self.logit_temp.exp().clamp(1e-2, 10.0) # uncomment when you make scale learnable

        # collapse dims → [?, T, d_k]
        if self.share_planes:
            # full collapse over M·B·H
            N = M * B * H
            Kh2 = Khf.permute(0,1,3,2,4).contiguous().view(N, T, dk)
            Qh2 = Qhf.permute(0,1,3,2,4).contiguous().view(N, T, dk)
            V2  = Vhf.permute(0,1,3,2,4).contiguous().view(N, T, dk)

            # single GEMM per projection
            projK = Kh2 @ self.planes_T                # [N, T, L*K]
            projQ = Qh2 @ self.planes_T                # [N, T, L*K]
        else:
            # collapse only batch+head per ensemble: [M, BH, T, d_k]
            BH = B * H
            Kh2 = Khf.permute(0,1,3,2,4).contiguous().view(M, BH, T, dk)
            Qh2 = Qhf.permute(0,1,3,2,4).contiguous().view(M, BH, T, dk)
            V2  = Vhf.permute(0,1,3,2,4).contiguous().view(M, BH, T, dk)

            # one batched GEMM across ensembles
            projK = torch.einsum('mbtd, mds -> mbts', Kh2, self.planes_T)
            projQ = torch.einsum('mbtd, mds -> mbts', Qh2, self.planes_T)
            # merge M,BH → N
            projK = projK.contiguous().view(M*BH, T, self.L*self.K)
            projQ = projQ.contiguous().view(M*BH, T, self.L*self.K)
            V2    = V2.view(M*BH, T, dk)
            N     = M * BH

        # reshape --> [N, T, L, K]
        projK = projK.view(N, T, self.L, self.K)
        projQ = projQ.view(N, T, self.L, self.K)

        # soft‑hash & bucket‑protos
        logitsK = (projK.tanh().div(scale) @ self.protos_T)  # [N, T, L, R]
        probsK  = F.softmax(logitsK, dim=-1)
        logitsQ = (projQ.tanh().div(scale) @ self.protos_T)
        probsQ  = F.softmax(logitsQ, dim=-1)

        # 2) causal prefix sums
        A_pref = probsK.cumsum(dim=1)                                    # [N, T, L, R]
        B_pref = (probsK.unsqueeze(-1) * V2.unsqueeze(2).unsqueeze(3)).cumsum(dim=1)                                       # [N, T, L, R, d_k]
        E_pref = B_pref.div(A_pref.unsqueeze(-1).add(1e-6))              # [N, T, L, R, d_k]

        S      = self.L * self.R

        # 3) lookup via one batched bmm
        out2 = torch.bmm(
            probsQ.view(N*T, 1, S),            # [N*T, 1, S]
            E_pref.contiguous().view(N*T, S, dk)            # [N*T, S, d_k]
        ).view(N, T, dk)                       # --> [N, T, d_k]

        # 4) reshape back --> [M, B, T, H, d_k]
        out = out2.view(M, B, H, T, dk).permute(0,1,3,2,4)
        return out

# Different
class RACEAttention(nn.Module):
    """Multi‑head wrapper around BatchedACE."""
    def __init__(self, d, h, K, L, M, drop=0.1,
                 qkv_bias=False, device='cpu'):
        super().__init__()
        assert d % h == 0
        self.h, self.d_k, self.M = h, d//h, M
        self.q = nn.Linear(d, d, bias=qkv_bias)
        self.k = nn.Linear(d, d, bias=qkv_bias)
        self.v = nn.Linear(d, d, bias=qkv_bias)
        self.o = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
        self.ace = BatchedACE(
                       self.d_k, K, L, M, device=device
                   )

    def forward(self, x, pad_mask=None):
        B, T, _ = x.shape
        # split heads
        Q = self.q(x).view(B, T, self.h, self.d_k)
        K = self.k(x).view(B, T, self.h, self.d_k)
        V = self.v(x).view(B, T, self.h, self.d_k)
        # pack M ensembles
        pack = lambda z: z.unsqueeze(0).expand(self.M, -1, -1, -1, -1)
        Khf, Vhf, Qhf = pack(K), pack(V), pack(Q)
        # single-shot causal ACE
        out_h = self.ace(Khf, Vhf, Qhf)       # [M, B, T, H, d_k]
        # merge ensembles & heads
        out   = out_h.mean(0).permute(0,2,1,3).contiguous().view(B, T, -1)
        return self.drop(self.o(out))


class RACEBlock(nn.Module):
    def __init__(self, cfg, device='cuda'):
        super().__init__()
        self.att = RACEAttention(
            d = cfg["emb_dim"],
            h = cfg["n_heads"],
            drop = cfg["drop_rate"],
            M = 2,          # number of ensembles
            K = 3,          # hash bits
            L = 2,          # number of hash tables
            qkv_bias = cfg.get("qkv_bias", False)
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            nn.GELU(),
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
        )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, pad_mask):
        h = x
        x = self.norm1(x)
        x = self.att(x, pad_mask)
        x = self.drop(x) + h

        h = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop(x) + h
        return x

class TransformerBlock(nn.Module):
    """Standard softmax‐attention Transformer block, pad‐mask aware."""
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(d=cfg["emb_dim"], h=cfg["n_heads"], drop=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"])
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                        nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
                        nn.GELU(),
                        nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
                     )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, pad_mask):
        h = x
        x = self.norm1(x)
        x = self.att(x, pad_mask)
        x = self.drop(x) + h

        h = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop(x) + h
        return x


class AngularBlock(nn.Module):
    """Angular (cosine)‐attention block, pad‐mask aware."""
    def __init__(self, cfg):
        super().__init__()
        self.att = AngularAttention(d=cfg["emb_dim"], h=cfg["n_heads"], drop=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"])

        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                        nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
                        nn.GELU(),
                        nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
                     )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, pad_mask):
        h = x
        x = self.norm1(x)
        x = self.att(x, pad_mask)
        x = self.drop(x) + h

        h = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop(x) + h
        return x

In [9]:
# ==================================================
# 8) Model & run_experiment
# ==================================================
class Classifier(nn.Module):
    def __init__(self, cfg, kind):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop    = nn.Dropout(cfg["drop_rate"])

        # Build the blocks list correctly
        self.blocks = nn.ModuleList()
        for _ in range(cfg["n_layers"]):
            if kind == "softmax":
                self.blocks.append( TransformerBlock(cfg) )
            elif kind == "angular":
                self.blocks.append( AngularBlock(cfg) )
            elif kind == "race":
                self.blocks.append( RACEBlock(cfg, device=DEVICE) )
            else:
                raise ValueError(kind)

        self.norm = nn.LayerNorm(cfg["emb_dim"])
        self.head = nn.Linear(cfg["emb_dim"], 2)

    def forward(self, x, mask):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.tok_emb(x) + self.pos_emb(pos)
        h = self.drop(h)
        for blk in self.blocks:
            h = blk(h, mask)
        h = self.norm(h)
        return self.head(h[:,0])



def run_experiment(attn_types, epochs=5, lr=1e-5, wd=5e-05):
    cfg = {
      "vocab_size": VOCAB_SIZE,
      "context_length": MAX_LEN,
      "emb_dim": 256,
      "n_heads": 2,
      "n_layers": 1,
      "drop_rate": 0.1,
      "qkv_bias": False,      # <— add this
    }


    for kind in attn_types:
        print(f"\n=== Training {kind.upper()} for {epochs} epochs ===")
        model = torch.compile(Classifier(cfg, kind)).to(DEVICE)
        opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        train_dl, test_dl = get_data()
        for ep in range(1, epochs+1):

            # --- train timing ---
            t0= time.time()
            model.train(); tl=ta=0
            for x,mask,y in train_dl:
                x,mask,y = x.to(DEVICE),mask.to(DEVICE),y.to(DEVICE)
                opt.zero_grad()
                logits = model(x,mask)
                loss   = F.cross_entropy(logits,y)
                acc    = (logits.argmax(-1)==y).float().mean().item()
                loss.backward(); opt.step()
                tl += loss.item(); ta += acc
            tr_l, tr_a = tl/len(train_dl), ta/len(train_dl)
            train_time = time.time() - t0

            # --- eval timing ---
            model.eval()
            t1 = time.time()
            vl=va=0
            with torch.no_grad():
                for x,mask,y in test_dl:
                    x,mask,y = x.to(DEVICE),mask.to(DEVICE),y.to(DEVICE)
                    logits = model(x,mask)
                    vl += F.cross_entropy(logits,y).item()
                    va += (logits.argmax(-1)==y).float().mean().item()
            va_l, va_a = vl/len(test_dl), va/len(test_dl)
            val_time = time.time() - t1

            print(
                f"Ep{ep:2d} | "
                f"train_loss {tr_l:.3f}, acc {tr_a:.3f} "
                f"({train_time:.1f}s) | "
                f"val_loss   {va_l:.3f}, acc {va_a:.3f} "
                f"({val_time:.1f}s)"
            )

In [10]:
run_experiment(["softmax", "race"], epochs=10)


=== Training SOFTMAX for 10 epochs ===
--> FINAL long_train: 11199
--> FINAL long_test:  2872


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
W1017 03:51:47.119000 530 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode


Ep 1 | train_loss 0.724, acc 0.506 (33.2s) | val_loss   0.713, acc 0.493 (13.8s)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Ep 2 | train_loss 0.711, acc 0.520 (15.8s) | val_loss   0.700, acc 0.515 (2.1s)
Ep 3 | train_loss 0.695, acc 0.541 (15.3s) | val_loss   0.678, acc 0.576 (2.2s)
Ep 4 | train_loss 0.676, acc 0.575 (15.9s) | val_loss   0.672, acc 0.590 (2.6s)
Ep 5 | train_loss 0.665, acc 0.597 (15.3s) | val_loss   0.671, acc 0.590 (2.1s)
Ep 6 | train_loss 0.659, acc 0.612 (15.4s) | val_loss   0.667, acc 0.597 (2.2s)
Ep 7 | train_loss 0.655, acc 0.610 (15.9s) | val_loss   0.671, acc 0.599 (2.3s)
Ep 8 | train_loss 0.649, acc 0.620 (15.3s) | val_loss   0.670, acc 0.602 (2.1s)
Ep 9 | train_loss 0.645, acc 0.624 (15.6s) | val_loss   0.659, acc 0.615 (2.6s)
Ep10 | train_loss 0.638, acc 0.637 (15.2s) | val_loss   0.666, acc 0.611 (2.1s)

=== Training RACE for 10 epochs ===
--> FINAL long_train: 11199
--> FINAL long_test:  2872
Ep 1 | train_loss 0.724, acc 0.497 (48.1s) | val_loss   0.703, acc 0.538 (11.1s)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Ep 2 | train_loss 0.708, acc 0.520 (23.3s) | val_loss   0.698, acc 0.533 (3.4s)
Ep 3 | train_loss 0.704, acc 0.517 (23.2s) | val_loss   0.694, acc 0.531 (3.0s)
Ep 4 | train_loss 0.701, acc 0.518 (23.4s) | val_loss   0.692, acc 0.541 (2.6s)
Ep 5 | train_loss 0.698, acc 0.524 (23.4s) | val_loss   0.690, acc 0.534 (2.5s)
Ep 6 | train_loss 0.696, acc 0.530 (23.5s) | val_loss   0.690, acc 0.542 (2.6s)
Ep 7 | train_loss 0.695, acc 0.527 (23.7s) | val_loss   0.689, acc 0.542 (2.6s)
Ep 8 | train_loss 0.691, acc 0.541 (23.7s) | val_loss   0.688, acc 0.543 (2.6s)
Ep 9 | train_loss 0.690, acc 0.539 (23.7s) | val_loss   0.689, acc 0.545 (2.5s)
Ep10 | train_loss 0.690, acc 0.536 (23.5s) | val_loss   0.687, acc 0.545 (2.6s)
